In [ ]:
%pip install -U trl

In [ ]:
from importlib import reload

from Trainers.trainer_ppo import PPOTrainingConfig, PolicyPPOTrainer
from Models.model_policy import PolicyModel
from Models.model_value import ValueModel
from Models.model_reward import RewardModel
import Datasets.dataset_request as dataset_request

dataset_request = reload(dataset_request)
RequestDataset = dataset_request.RequestDataset

In [ ]:

config = PPOTrainingConfig(
    output_dir="outputs/ppo_policy"
)
policy = PolicyModel("Qwen/Qwen3-0.6B")
value = ValueModel("Qwen/Qwen3-0.6B")
reward_model = RewardModel("Skywork/Skywork-Reward-V2-Qwen3-0.6B", "proxy")
dataset = RequestDataset.load("human_requests_hh-rlhf.pt", "Qwen/Qwen3-0.6B")
data_list = dataset.get(0,2)
dataset.truncate(0, 2)

In [ ]:
answers = policy.generate_batch(data_list)
for question, answer in zip(data_list, answers):
    print(f"Question:\n {question}\n\nAnswer:\n {answer}\n\n")
    

In [ ]:
%pip show trl

In [ ]:
trainer = PolicyPPOTrainer(policy, reward_model, value, dataset, config)

In [ ]:
trainer.train()

In [ ]:
# After training, policy is changed. I want to answers = policy.generate_batch(data_list)
answers = policy.generate_batch(data_list)
for question, answer in zip(data_list, answers):
    print(f"Question:\n {question}\n\nAnswer:\n {answer}\n\n")

In [ ]:
from Models.model_evaluator import PrometheusEvaluator


evaluator = PrometheusEvaluator()
print(evaluator.score(data_list[0], answers[0]))